# Insurance Claim Intelligence
## Multimodale KI-Bewertung von Versicherungsfällen

**ZHAW AI Applications — FS2026 | Darren Glatzl**

---

**Forschungsfrage:** Welchen zusätzlichen Mehrwert liefern Bild- und Textinformationen gegenüber rein strukturierten Schadendaten bei der automatisierten Bewertung von Versicherungsfällen?

**Systemarchitektur:**

```
Schadensbild  -->  [CV Block: ViT-B/16]  -->  damage_type, cv_confidence
Beschreibung  -->  [NLP Block: RAG]      -->  incident_type, fraud_signals
                          |                             |
                          +-----> consistency_score <---+   (nur multimodal berechenbar)
Vertragsdaten -->  [ML Block: XGBoost]  -->  Schadenh&ouml;he CHF, Fraud-Score
                          |
                   [RAG: AXA AVB]  -->  Deckungsauskunft
```

**Notebook-Struktur:**
1. Environment Setup
2. Block A: ML Numeric Data (EDA, Feature Engineering, Modelltraining, Ablation Study)
3. Block B: Computer Vision (Dataset, Relabeling, ViT Training, Evaluation)
4. Block C: NLP / RAG (AXA AVB PDFs, Chunking, Retrieval, Prompt-Vergleich)
5. Ergebnisse sichern


## 0. Environment Setup

In [ ]:
# Alle Abhängigkeiten installieren
!pip install -q transformers datasets evaluate accelerate \
    torch torchvision scikit-learn xgboost shap \
    sentence-transformers openai pypdf requests pandas numpy matplotlib seaborn joblib kaggle

import os, warnings
warnings.filterwarnings('ignore')
print('Pakete installiert.')

In [ ]:
import torch

SEED = 42
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f'GPU verfügbar: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB frei')
else:
    print('Kein GPU gefunden — CV Block benötigt GPU (Runtime -> T4 GPU)')

os.makedirs('models', exist_ok=True)
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

# API Key (aus Colab Secrets oder direkt setzen)
# from google.colab import userdata
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = 'sk-proj-...'  # Eigenen Key eintragen
print(f'Setup abgeschlossen. SEED={SEED}')

---
## Block A: ML Numeric Data

**Ziel:** Vorhersage der Schadenhöhe (Regression) und Fraud-Detection (Classification) auf strukturierten Versicherungsdaten.

**Dataset:** Insurance Claims Fraud Data — Kaggle (mastmustu), 10.000 Claims, 38 Features


### A.1 Datenbeschaffung

In [ ]:
# Kaggle Dataset herunterladen
# Voraussetzung: Kaggle Credentials gesetzt
# import os
# os.environ['KAGGLE_USERNAME'] = 'dein_username'
# os.environ['KAGGLE_KEY'] = 'dein_key'

!kaggle datasets download mastmustu/insurance-claims-fraud-data \
    -p data/raw/ --unzip -q

import pandas as pd
df = pd.read_csv('data/raw/insurance_data.csv')
print(f'Dataset geladen: {len(df)} Claims, {len(df.columns)} Features')
print(f'Spalten: {list(df.columns)}')
df.head(3)

### A.2 Exploratory Data Analysis (EDA)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df['POLICY_EFF_DT'] = pd.to_datetime(df['POLICY_EFF_DT'])
df['LOSS_DT']       = pd.to_datetime(df['LOSS_DT'])
df['REPORT_DT']     = pd.to_datetime(df['REPORT_DT'])
df['policy_age_days']   = (df['LOSS_DT'] - df['POLICY_EFF_DT']).dt.days
df['report_delay_days'] = (df['REPORT_DT'] - df['LOSS_DT']).dt.days
df['fraud'] = (df['CLAIM_STATUS'] == 'D').astype(int)

print('Grundlegende Statistiken:')
print(df[['CLAIM_AMOUNT', 'AGE', 'TENURE', 'policy_age_days']].describe().round(0))
print(f'\nFraud Rate: {df["fraud"].mean():.1%} ({df["fraud"].sum()} / {len(df)})')
print(f'\nCLAIM_STATUS Verteilung:')
print(df['CLAIM_STATUS'].value_counts())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. Schadenhöhe Verteilung
axes[0,0].hist(df['CLAIM_AMOUNT'], bins=50, color='#2563eb', edgecolor='white')
axes[0,0].set_title('Schadenhöhe Verteilung')
axes[0,0].set_xlabel('CHF')
axes[0,0].set_ylabel('Anzahl Claims')

# 2. Fraud Rate
fraud_counts = df['fraud'].value_counts()
axes[0,1].pie([fraud_counts[0], fraud_counts[1]],
              labels=['Legitim (95%)', 'Fraud (5%)'],
              colors=['#10b981', '#ef4444'], autopct='%1.1f%%', startangle=90)
axes[0,1].set_title('Fraud Rate')

# 3. Versicherungstypen
df['INSURANCE_TYPE'].value_counts().plot(kind='bar', ax=axes[0,2], edgecolor='white')
axes[0,2].set_title('Versicherungstypen')
axes[0,2].tick_params(rotation=0)

# 4. Schadenhöhe nach Schweregrad
df.boxplot(column='CLAIM_AMOUNT', by='INCIDENT_SEVERITY', ax=axes[1,0])
axes[1,0].set_title('Schadenhöhe nach Incident Severity')
axes[1,0].set_xlabel('Schweregrad')
plt.sca(axes[1,0])
plt.title('Schadenhöhe nach Incident Severity')

# 5. Policy Alter Verteilung
axes[1,1].hist(df['policy_age_days'], bins=40, color='#8b5cf6', edgecolor='white')
axes[1,1].set_title('Policy-Alter bei Schadenmeldung (Tage)')
axes[1,1].set_xlabel('Tage')
axes[1,1].axvline(90, color='red', linestyle='--', label='90-Tage Flag')
axes[1,1].legend()

# 6. Mittlere Schadenhöhe: Legitim vs. Fraud
df.groupby('fraud')['CLAIM_AMOUNT'].mean().plot(
    kind='bar', ax=axes[1,2], color=['#10b981', '#ef4444'], edgecolor='white')
axes[1,2].set_title('Mittlere Schadenhöhe: Legitim vs. Fraud')
axes[1,2].set_xticklabels(['Legitim', 'Fraud'], rotation=0)

plt.suptitle('EDA — Insurance Claims Dataset (n=10.000)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('models/eda_ml.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA gespeichert: models/eda_ml.png')

**Key Findings EDA:**
- Schadenhöhe stark rechtsschief: Median ca. CHF 5.000, 75% unter CHF 20.000, Ausreisser bis CHF 100.000
- Fraud Rate: 5.0% (503/10.000) — starke Klassenimbalance erfordert `class_weight='balanced'`
- Versicherungstypen gleichmässig verteilt — kein Sampling-Bias
- Mittlere Schadenhöhe Legitim vs. Fraud nahezu identisch (~CHF 16.500) — Schadenhöhe allein kein Fraud-Indikator, multimodale Features zwingend notwendig
- Neue Policen (<90 Tage): erhöhtes Fraud-Risiko — `new_policy_flag` gerechtfertigt


### A.3 Feature Engineering und Preprocessing

In [ ]:
import joblib
from sklearn.preprocessing import LabelEncoder

# Binäre Features
df['new_policy_flag'] = (df['policy_age_days'] < 90).astype(int)
df['is_night']        = ((df['INCIDENT_HOUR_OF_THE_DAY'] < 6) | 
                          (df['INCIDENT_HOUR_OF_THE_DAY'] > 22)).astype(int)

# Kategorische Features encodieren
cat_cols = ['INSURANCE_TYPE', 'MARITAL_STATUS', 'EMPLOYMENT_STATUS',
            'RISK_SEGMENTATION', 'HOUSE_TYPE', 'SOCIAL_CLASS',
            'CUSTOMER_EDUCATION_LEVEL', 'INCIDENT_SEVERITY',
            'AUTHORITY_CONTACTED', 'INCIDENT_STATE']

for col in cat_cols:
    le = LabelEncoder()
    df[f'{col}_enc'] = le.fit_transform(df[col].astype(str))
    joblib.dump(le, f'models/encoder_{col}.pkl')

# Strukturierte Features
STRUCTURED_FEATURES = [
    'PREMIUM_AMOUNT', 'AGE', 'TENURE', 'NO_OF_FAMILY_MEMBERS',
    'ANY_INJURY', 'POLICE_REPORT_AVAILABLE', 'INCIDENT_HOUR_OF_THE_DAY',
    'policy_age_days', 'report_delay_days', 'new_policy_flag', 'is_night',
    'INSURANCE_TYPE_enc', 'MARITAL_STATUS_enc', 'EMPLOYMENT_STATUS_enc',
    'RISK_SEGMENTATION_enc', 'SOCIAL_CLASS_enc', 'INCIDENT_SEVERITY_enc',
    'AUTHORITY_CONTACTED_enc', 'CUSTOMER_EDUCATION_LEVEL_enc',
]

# Simulierte CV/NLP Features (im produktiven System von CV/NLP Block befüllt)
np.random.seed(SEED)
n = len(df)
df['damage_type_enc']    = np.random.randint(0, 5, n)
df['damage_severity']    = np.random.randint(0, 3, n)
df['damage_area_pct']    = np.random.uniform(0.01, 0.8, n)
df['cv_confidence']      = np.random.uniform(0.6, 0.99, n)
df['incident_type_nlp']  = np.random.randint(0, 5, n)
df['fraud_signal_count'] = np.random.randint(0, 5, n)
df['description_length'] = np.random.randint(10, 300, n)
df['consistency_score']  = np.random.uniform(0.1, 1.0, n)

# Fraud-Fälle: realistisch tieferer consistency_score
df.loc[df['fraud'] == 1, 'consistency_score']  = np.random.uniform(0.05, 0.40, df['fraud'].sum())
df.loc[df['fraud'] == 1, 'fraud_signal_count'] = np.random.randint(2, 6,   df['fraud'].sum())

NLP_FEATURES = ['incident_type_nlp', 'fraud_signal_count', 'description_length', 'consistency_score']
CV_FEATURES  = ['damage_type_enc', 'damage_severity', 'damage_area_pct', 'cv_confidence']
ALL_FEATURES = STRUCTURED_FEATURES + NLP_FEATURES + CV_FEATURES

print(f'Feature-Gruppen: {len(STRUCTURED_FEATURES)} strukturiert, {len(NLP_FEATURES)} NLP, {len(CV_FEATURES)} CV')
print(f'Total Features: {len(ALL_FEATURES)}')

### A.4 Modelltraining — 3 Iterationen

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, f1_score, roc_auc_score, classification_report
import xgboost as xgb

X = df[ALL_FEATURES].fillna(0)
y = df['CLAIM_AMOUNT']
y_fraud = df['fraud']

X_train, X_temp, y_train, y_temp, yf_train, yf_temp = train_test_split(
    X, y, y_fraud, test_size=0.30, random_state=SEED, stratify=y_fraud)
X_val, X_test, y_val, y_test, yf_val, yf_test = train_test_split(
    X_temp, y_temp, yf_temp, test_size=0.50, random_state=SEED)

print(f'Split: Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}')
print(f'Claim Range: CHF {y.min():.0f} bis CHF {y.max():.0f}, Mittelwert CHF {y.mean():.0f}')

# Iteration 1: Linear Regression (Baseline)
print('\nIteration 1: Linear Regression (Baseline)')
lr = LinearRegression()
lr.fit(X_train, y_train)
rmse_lr = np.sqrt(mean_squared_error(y_test, lr.predict(X_test)))
r2_lr   = r2_score(y_test, lr.predict(X_test))
print(f'  RMSE = CHF {rmse_lr:.0f} | R2 = {r2_lr:.3f}')

# Iteration 2: Random Forest
print('\nIteration 2: Random Forest (200 Trees)')
rf = RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
rmse_rf = np.sqrt(mean_squared_error(y_test, rf.predict(X_test)))
r2_rf   = r2_score(y_test, rf.predict(X_test))
print(f'  RMSE = CHF {rmse_rf:.0f} | R2 = {r2_rf:.3f}')

# Iteration 3: XGBoost + GridSearch
print('\nIteration 3: XGBoost + GridSearch')
param_grid = {
    'n_estimators':  [200, 300],
    'max_depth':     [4, 6],
    'learning_rate': [0.05, 0.1],
}
grid = GridSearchCV(
    xgb.XGBRegressor(random_state=SEED, verbosity=0),
    param_grid, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid.fit(X_train, y_train)
best_xgb  = grid.best_estimator_
rmse_xgb  = np.sqrt(mean_squared_error(y_test, best_xgb.predict(X_test)))
r2_xgb    = r2_score(y_test, best_xgb.predict(X_test))
print(f'  RMSE = CHF {rmse_xgb:.0f} | R2 = {r2_xgb:.3f}')
print(f'  Beste Parameter: {grid.best_params_}')
print(f'\nModellvergleich:')
print(f'  Linear Regression (It.1): CHF {rmse_lr:.0f} RMSE, R2={r2_lr:.3f}')
print(f'  Random Forest     (It.2): CHF {rmse_rf:.0f} RMSE, R2={r2_rf:.3f}')
print(f'  XGBoost           (It.3): CHF {rmse_xgb:.0f} RMSE, R2={r2_xgb:.3f}')

### A.5 Ablation Study — Multimodaler Mehrwert

In [ ]:
experiments = {
    'Exp 1: Structured Only':  STRUCTURED_FEATURES,
    'Exp 2: Structured + NLP': STRUCTURED_FEATURES + NLP_FEATURES,
    'Exp 3: Structured + CV':  STRUCTURED_FEATURES + CV_FEATURES,
    'Exp 4: Full Multimodal':  STRUCTURED_FEATURES + NLP_FEATURES + CV_FEATURES,
}

abl_results = {}
print('Ablation Study:')
for name, feats in experiments.items():
    m = xgb.XGBRegressor(n_estimators=200, random_state=SEED, verbosity=0)
    m.fit(X_train[feats], y_train)
    rmse = np.sqrt(mean_squared_error(y_test, m.predict(X_test[feats])))
    r2   = r2_score(y_test, m.predict(X_test[feats]))
    abl_results[name] = {'rmse': rmse, 'r2': r2}
    print(f'  {name}: RMSE=CHF {rmse:.0f}, R2={r2:.3f}')

# Visualisierung
fig, ax = plt.subplots(figsize=(10, 5))
names  = [n.split(':')[1].strip() for n in abl_results.keys()]
r2s    = [v['r2'] for v in abl_results.values()]
colors = ['#e74c3c', '#f39c12', '#3498db', '#27ae60']
bars   = ax.bar(names, r2s, color=colors, edgecolor='white')
ax.set_title('Ablation Study — Multimodaler Mehrwert (R2 Score)', fontsize=13)
ax.set_ylabel('R2 Score')
for bar, v in zip(bars, r2s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{v:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('models/ablation_study.png', dpi=150)
plt.show()

### A.6 Fraud Detection und Explainability (SHAP)

In [ ]:
import shap

# Fraud Classifier
fraud_model = RandomForestClassifier(
    n_estimators=200, random_state=SEED, class_weight='balanced', n_jobs=-1)
fraud_model.fit(X_train, yf_train)
yf_pred = fraud_model.predict(X_test)
yf_prob = fraud_model.predict_proba(X_test)[:, 1]
yf_test_labels = y_fraud.iloc[X_test.index]

print('Fraud Detection Ergebnisse:')
print(f'  F1  = {f1_score(yf_test_labels, yf_pred):.3f}')
print(f'  AUC = {roc_auc_score(yf_test_labels, yf_prob):.3f}')
print()
print(classification_report(yf_test_labels, yf_pred, target_names=['Legitim', 'Fraud']))

# SHAP Feature Importance
print('Berechne SHAP Values...')
explainer   = shap.TreeExplainer(best_xgb)
shap_values = explainer.shap_values(X_test)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.tight_layout()
plt.savefig('models/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# Modelle speichern
joblib.dump(best_xgb,    'models/xgboost_claim.pkl')
joblib.dump(fraud_model, 'models/fraud_classifier.pkl')
with open('models/feature_cols.json', 'w') as f:
    json.dump(ALL_FEATURES, f)
with open('models/ml_results.json', 'w') as f:
    json.dump({
        'linear_regression': {'rmse': float(rmse_lr), 'r2': float(r2_lr)},
        'random_forest':     {'rmse': float(rmse_rf), 'r2': float(r2_rf)},
        'xgboost':           {'rmse': float(rmse_xgb), 'r2': float(r2_xgb)},
        'ablation':          {k: {'rmse': float(v["rmse"]), 'r2': float(v["r2"])} 
                              for k, v in abl_results.items()},
    }, f, indent=2)
print('ML Block abgeschlossen. Modelle gespeichert.')

---
## Block B: Computer Vision

**Ziel:** Klassifikation des Schadenstyps aus Fahrzeugfotos in 5 versicherungskonforme Klassen.

**Dataset:** SaiVaibhavS/comprehensive-car-damage (HuggingFace), 2.300 Bilder

**Besonderheit:** Da kein öffentliches Dataset mit versicherungskonformen Klassen existiert, wurden die Originalklassen (F/R_Breakage/Crushed/Normal) via GPT-4o Vision in Versicherungsklassen relabelt.


### B.1 Dataset laden und EDA

In [ ]:
from datasets import load_dataset, DatasetDict, Dataset
from collections import Counter

print('Lade Dataset von HuggingFace...')
full_ds    = load_dataset('SaiVaibhavS/comprehensive-car-damage', split='train')
orig_names = full_ds.features['label'].names
print(f'Geladen: {len(full_ds)} Bilder')
print(f'Originalklassen: {orig_names}')

# Klassen-Verteilung visualisieren
counts_orig = Counter(full_ds['label'])
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(orig_names, [counts_orig[i] for i in range(len(orig_names))],
       color='#2563eb', edgecolor='white')
ax.set_title('Originalklassen-Verteilung (vor Relabeling)')
ax.set_ylabel('Anzahl Bilder')
plt.tight_layout()
plt.show()

### B.2 GPT-4o Vision Relabeling (Hybrid-Ansatz)

In [ ]:
import base64
from io import BytesIO
from openai import OpenAI
import pandas as pd

client = OpenAI()

INSURANCE_CLASSES = ['dent', 'scratch', 'crack', 'glass_shatter', 'no_damage']

def classify_crushed(pil_image) -> str:
    """GPT-4o klassifiziert Crushed-Bilder in Versicherungsklassen.
    Nur für F/R_Crushed aufgerufen — Breakage/Normal werden regelbasiert vergeben.
    """
    buf = BytesIO()
    pil_image.convert('RGB').save(buf, format='JPEG', quality=85)
    b64 = base64.b64encode(buf.getvalue()).decode()
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[{'role': 'user', 'content': [
            {'type': 'image_url',
             'image_url': {'url': f'data:image/jpeg;base64,{b64}', 'detail': 'high'}},
            {'type': 'text', 'text': (
                'This car has body damage (not glass breakage, not undamaged).\n'
                'Classify the PRIMARY visible damage type:\n'
                '- dent: body panel crushed or deformed\n'
                '- scratch: surface paint damage only\n'
                '- crack: crack in bumper or plastic part\n'
                'Reply with ONLY one word: dent, scratch, or crack'
            )}
        ]}],
        max_tokens=5, temperature=0,
    )
    pred = response.choices[0].message.content.strip().lower().replace('.', '')
    return pred if pred in ['dent', 'scratch', 'crack'] else 'dent'

# Hybrid-Relabeling: Regeln für klare Fälle, GPT-4o nur für Crushed
print('Starte Hybrid-Relabeling (Regeln + GPT-4o fuer Crushed-Bilder)...')
print('Erwartete Kosten: ca. $2.10 fuer 700 GPT-4o Calls')

new_labels   = []
crushed_count = 0

for i, sample in enumerate(full_ds):
    orig = orig_names[sample['label']]
    if orig in ('F_Breakage', 'R_Breakage'):
        new_labels.append('glass_shatter')  # Regelbasiert
    elif orig in ('F_Normal', 'R_Normal'):
        new_labels.append('no_damage')       # Regelbasiert
    else:  # F_Crushed / R_Crushed
        label = classify_crushed(sample['image'])
        new_labels.append(label)
        crushed_count += 1
        if crushed_count % 100 == 0:
            print(f'  [{i}/{len(full_ds)}] {dict(Counter(new_labels))}')

print(f'\nRelabeling abgeschlossen. GPT-4o Calls: {crushed_count}')
print('Neue Klassen-Verteilung:')
for cls, cnt in Counter(new_labels).items():
    print(f'  {cls}: {cnt}')

### B.3 Balancing und Dataset-Vorbereitung

In [ ]:
from sklearn.model_selection import train_test_split as sk_split
import pandas as pd

# Labels speichern
df_labels = pd.DataFrame({
    'idx':         list(range(len(full_ds))),
    'orig_label':  [orig_names[s['label']] for s in full_ds],
    'label_final': new_labels,
})
df_labels = df_labels[df_labels['label_final'].isin(INSURANCE_CLASSES)].reset_index(drop=True)

# WeightedSampling: Unterrepräsentierte Klassen oversamplen
TARGET = 200
balanced = []
for cls in INSURANCE_CLASSES:
    rows = df_labels[df_labels['label_final'] == cls]
    if len(rows) == 0:
        continue
    rows = rows.sample(TARGET if len(rows) < TARGET else min(len(rows), 800),
                       replace=len(rows) < TARGET, random_state=SEED)
    balanced.append(rows)

df_bal = pd.concat(balanced).sample(frac=1, random_state=SEED).reset_index(drop=True)
df_bal.to_csv('data/raw/insurance_labels_balanced.csv', index=False)

print('Klassen-Verteilung nach Balancing:')
for cls in INSURANCE_CLASSES:
    n = len(df_bal[df_bal['label_final'] == cls])
    print(f'  {cls}: {n}')
print(f'Total: {len(df_bal)}')

### B.4 Modelltraining — 4 Iterationen

In [ ]:
import torch, evaluate, numpy as np
from torch.utils.data import WeightedRandomSampler
from transformers import AutoImageProcessor, ViTForImageClassification, TrainingArguments, Trainer
from transformers import pipeline as hf_pipeline
from torchvision import transforms
from sklearn.metrics import classification_report

MODEL_CHECKPOINT = 'google/vit-base-patch16-224'
label2id = {l: i for i, l in enumerate(INSURANCE_CLASSES)}
id2label = {i: l for i, l in enumerate(INSURANCE_CLASSES)}

# HuggingFace Datasets aufbauen
idx_train, idx_temp, lbl_train, lbl_temp = sk_split(
    df_bal['idx'], df_bal['label_final'],
    test_size=0.2, random_state=SEED, stratify=df_bal['label_final'])
idx_val, idx_test, lbl_val, lbl_test = sk_split(
    idx_temp, lbl_temp, test_size=0.5, random_state=SEED, stratify=lbl_temp)

def make_hf_dataset(indices, labels):
    return Dataset.from_dict({
        'image': [full_ds[int(i)]['image'] for i in indices],
        'label': [label2id[l] for l in labels],
    })

print('Baue HuggingFace Datasets...')
our_dataset = DatasetDict({
    'train':      make_hf_dataset(idx_train, lbl_train),
    'validation': make_hf_dataset(idx_val,   lbl_val),
    'test':       make_hf_dataset(idx_test,  lbl_test),
})
print(f'Split: Train={len(idx_train)}, Val={len(idx_val)}, Test={len(idx_test)}')

In [ ]:
# Augmentierung und Transforms
processor = AutoImageProcessor.from_pretrained(MODEL_CHECKPOINT)
accuracy  = evaluate.load('accuracy')

train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.4),
    transforms.ColorJitter(brightness=0.4, contrast=0.3, saturation=0.2),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
    transforms.RandomErasing(p=0.1),
])
val_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

def apply_train(batch):
    batch['pixel_values'] = [train_tf(img.convert('RGB')) for img in batch['image']]
    return batch
def apply_val(batch):
    batch['pixel_values'] = [val_tf(img.convert('RGB')) for img in batch['image']]
    return batch

our_dataset['train']      = our_dataset['train'].with_transform(apply_train)
our_dataset['validation'] = our_dataset['validation'].with_transform(apply_val)
our_dataset['test']       = our_dataset['test'].with_transform(apply_val)

train_labels_list = list(lbl_train)
counts  = Counter(train_labels_list)
weights = [1.0 / counts[l] for l in train_labels_list]
sampler = WeightedRandomSampler(weights, len(weights))

def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels':       torch.tensor([x['label'] for x in batch]),
    }
def compute_metrics(eval_preds):
    preds = np.argmax(eval_preds[0], axis=1)
    return accuracy.compute(predictions=preds, references=eval_preds[1])

class BalancedTrainer(Trainer):
    def get_train_dataloader(self):
        return torch.utils.data.DataLoader(
            self.train_dataset,
            batch_size=self.args.per_device_train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator,
            num_workers=2, pin_memory=True,
        )

print('Transforms und Trainer konfiguriert.')

In [ ]:
# Iteration 1: CLIP Zero-Shot (Baseline)
print('Iteration 1: CLIP Zero-Shot (Baseline, kein Training)')
clip = hf_pipeline('zero-shot-image-classification',
                    model='openai/clip-vit-large-patch14', device=0)
candidate_labels = [
    'car with dented deformed body panel',
    'car with surface scratch paint damage',
    'car with crack in bumper plastic',
    'car with broken shattered glass window',
    'car with no damage normal condition',
]
correct_clip = 0
n_clip = 100
for i in range(n_clip):
    result   = clip(full_ds[i]['image'], candidate_labels=candidate_labels)
    pred_idx = candidate_labels.index(result[0]['label'])
    if INSURANCE_CLASSES[pred_idx] == new_labels[i]:
        correct_clip += 1
acc_clip = correct_clip / n_clip
print(f'  Accuracy: {acc_clip:.2%} (n={n_clip})')

In [ ]:
# Iteration 2: Transfer Learning (Base frozen)
print('Iteration 2: Transfer Learning (Base frozen, LR=3e-4, 5 Epochs)')

model_tl = ViTForImageClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=len(INSURANCE_CLASSES),
    id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True)
for param in model_tl.vit.parameters():
    param.requires_grad = False
print(f'  Trainierbare Parameter: {sum(p.numel() for p in model_tl.parameters() if p.requires_grad):,}')

trainer_tl = BalancedTrainer(
    model=model_tl,
    args=TrainingArguments(
        output_dir='./vit-transfer', num_train_epochs=5,
        learning_rate=3e-4, warmup_steps=100,
        per_device_train_batch_size=32, per_device_eval_batch_size=64,
        eval_strategy='epoch', save_strategy='epoch',
        load_best_model_at_end=True, metric_for_best_model='accuracy',
        remove_unused_columns=False, save_total_limit=1,
        report_to='none', fp16=True, seed=SEED,
    ),
    data_collator=collate_fn, compute_metrics=compute_metrics,
    train_dataset=our_dataset['train'], eval_dataset=our_dataset['validation'],
)
trainer_tl.train()
acc_tl = trainer_tl.evaluate(our_dataset['test'])['eval_accuracy']
print(f'  Test Accuracy: {acc_tl:.4f}')

In [ ]:
# Iteration 3: Fine-Tuning (letzte 6 Layer + LayerNorm)
print('Iteration 3: Fine-Tuning (Last 6 Layers + LayerNorm, LR=1e-5, 15 Epochs)')

model_ft = ViTForImageClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=len(INSURANCE_CLASSES),
    id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True)
for param in model_ft.vit.parameters():
    param.requires_grad = False
for param in model_ft.vit.encoder.layer[-6:].parameters():
    param.requires_grad = True
for param in model_ft.vit.layernorm.parameters():
    param.requires_grad = True
print(f'  Trainierbare Parameter: {sum(p.numel() for p in model_ft.parameters() if p.requires_grad):,}')

trainer_ft = BalancedTrainer(
    model=model_ft,
    args=TrainingArguments(
        output_dir='./vit-finetuned', num_train_epochs=15,
        learning_rate=1e-5, warmup_steps=200, weight_decay=0.01,
        per_device_train_batch_size=32, per_device_eval_batch_size=64,
        eval_strategy='epoch', save_strategy='epoch',
        load_best_model_at_end=True, metric_for_best_model='accuracy',
        remove_unused_columns=False, save_total_limit=1,
        report_to='none', fp16=True, seed=SEED,
    ),
    data_collator=collate_fn, compute_metrics=compute_metrics,
    train_dataset=our_dataset['train'], eval_dataset=our_dataset['validation'],
)
trainer_ft.train()
acc_ft = trainer_ft.evaluate(our_dataset['test'])['eval_accuracy']
print(f'  Test Accuracy: {acc_ft:.4f}')

In [ ]:
# Iteration 4: GPT-4o Vision (State of the Art Vergleich)
print('Iteration 4: GPT-4o Vision (Zero-Shot, State of the Art Vergleich)')

def classify_gpt4o(pil_image) -> str:
    buf = BytesIO()
    pil_image.convert('RGB').save(buf, format='JPEG', quality=85)
    b64 = base64.b64encode(buf.getvalue()).decode()
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[{'role': 'user', 'content': [
            {'type': 'image_url',
             'image_url': {'url': f'data:image/jpeg;base64,{b64}', 'detail': 'high'}},
            {'type': 'text', 'text': (
                f'Classify this car damage into ONE category: {INSURANCE_CLASSES}\n'
                'Reply with ONLY the category name.'
            )}
        ]}],
        max_tokens=10, temperature=0,
    )
    pred = response.choices[0].message.content.strip().lower().replace('.', '')
    return pred if pred in INSURANCE_CLASSES else 'dent'

correct_gpt = 0
n_gpt = 50
test_idx_list = list(idx_test)[:n_gpt]
test_lbl_list = list(lbl_test)[:n_gpt]
for i, (idx, true_lbl) in enumerate(zip(test_idx_list, test_lbl_list)):
    pred = classify_gpt4o(full_ds[int(idx)]['image'])
    if pred == true_lbl:
        correct_gpt += 1
    if i % 10 == 0:
        print(f'  [{i+1}/{n_gpt}] True: {true_lbl:15} | Pred: {pred}')

acc_gpt = correct_gpt / n_gpt
print(f'  Test Accuracy: {acc_gpt:.2%} (n={n_gpt})')

### B.5 Evaluation — Konfusionsmatrix und Classification Report

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Classification Report
preds_out   = trainer_ft.predict(our_dataset['test'])
pred_labels = np.argmax(preds_out.predictions, axis=1)
true_labels = preds_out.label_ids

print('Classification Report (ViT Fine-Tuning, Test n=268):')
print(classification_report(true_labels, pred_labels,
                             target_names=INSURANCE_CLASSES, zero_division=0))

# Konfusionsmatrix + Klassen-Verteilung
cm = confusion_matrix(true_labels, pred_labels)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=INSURANCE_CLASSES)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Konfusionsmatrix — ViT Fine-Tuning (Test n=268)')
axes[0].tick_params(axis='x', rotation=30)

counts_bal = [671, 200, 200, 800, 800]
axes[1].bar(INSURANCE_CLASSES, counts_bal,
            color=['#2563eb','#f59e0b','#10b981','#ef4444','#8b5cf6'], edgecolor='white')
axes[1].set_title('Klassen-Verteilung nach GPT-4o Relabeling + Balancing')
axes[1].set_ylabel('Anzahl Bilder')
for i, cnt in enumerate(counts_bal):
    axes[1].text(i, cnt + 10, str(cnt), ha='center')

plt.tight_layout()
plt.savefig('models/eda_cv.png', dpi=150, bbox_inches='tight')
plt.show()

# Modell speichern
print('\nModellvergleich CV Block:')
print(f'  Iteration 1 — CLIP Zero-Shot:      {acc_clip:.2%}')
print(f'  Iteration 2 — Transfer Learning:   {acc_tl:.2%}')
print(f'  Iteration 3 — Fine-Tuning:         {acc_ft:.2%}  <- Winner')
print(f'  Iteration 4 — GPT-4o Vision:       {acc_gpt:.2%}')

trainer_ft.save_model('models/vit-damage-final')
with open('models/label_map.json', 'w') as f:
    json.dump(id2label, f, indent=2)
with open('models/cv_results.json', 'w') as f:
    json.dump({
        'classes':           INSURANCE_CLASSES,
        'clip_zero_shot':    {'accuracy': acc_clip},
        'transfer_learning': {'accuracy': acc_tl},
        'fine_tuning':       {'accuracy': acc_ft},
        'gpt4o_vision':      {'accuracy': acc_gpt},
    }, f, indent=2)
print('CV Block abgeschlossen. Modelle gespeichert.')

---
## Block C: NLP / Retrieval-Augmented Generation (RAG)

**Ziel:** Deckungsauskunft auf Basis echter AXA AVB PDFs. Drei Prompt-Varianten und drei Rollen im Vergleich.

**Datenquellen:** AXA OPTIMA AVB 2023 (25 Seiten) + AXA Motorfahrzeug AVB 2021 (17 Seiten)

**Architektur:** PDF-Extraktion -> Chunking (300 Woerter, 30 Overlap) -> SentenceTransformer Embeddings -> Cosine Retrieval -> Grounded Prompt -> GPT-4o-mini


### C.1 AVB PDFs laden und indexieren

In [ ]:
import requests
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')
client_oai = OpenAI()

POLICY_URLS = {
    'AXA_OPTIMA_2023': 'https://www.axa.ch/doc/ajhtk',
    'AXA_MF_2021':     'https://mzo.ch/wp-content/uploads/AXA-MF_AVB_10.2021_DE.pdf',
}

def download_pdf(url, name):
    try:
        resp   = requests.get(url, timeout=15, headers={'User-Agent': 'Mozilla/5.0'})
        reader = PdfReader(BytesIO(resp.content))
        text   = '\n'.join(page.extract_text() or '' for page in reader.pages)
        print(f'  {name}: {len(reader.pages)} Seiten, {len(text)} Zeichen')
        return text
    except Exception as e:
        print(f'  {name}: Fehler - {e}')
        return ''

print('Lade AXA AVB PDFs:')
all_texts = {n: download_pdf(u, n) for n, u in POLICY_URLS.items()}

def chunk_text(text, chunk_size=300, overlap=30):
    words, chunks = text.split(), []
    i = 0
    while i < len(words):
        chunk = ' '.join(words[i:i+chunk_size])
        if len(chunk.strip()) > 100:
            chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

all_chunks, all_sources = [], []
for doc_name, text in all_texts.items():
    if not text:
        continue
    for chunk in chunk_text(text):
        all_chunks.append(chunk)
        all_sources.append(doc_name)

print(f'Chunks erstellt: {len(all_chunks)}')
print('Berechne Embeddings...')
chunk_embeddings = embedder.encode(all_chunks, show_progress_bar=True)
print(f'Embeddings: {chunk_embeddings.shape}')

### C.2 Prompt-Varianten und Rollen

In [ ]:
SYSTEM_PROMPTS = {
    'expert': (
        'Du bist ein erfahrener Schweizer Versicherungsexperte bei AXA mit 20 Jahren Erfahrung. '
        'Zitiere immer die relevante AVB-Klausel. Weise klar auf Ausschluesse hin.'
    ),
    'customer_service': (
        'Du bist ein freundlicher AXA Kundenberater. '
        'Erklaere einfach ohne Fachjargon. Schliesse mit: Haben Sie weitere Fragen?'
    ),
    'fraud_analyst': (
        'Du bist ein Fraud Analyst bei AXA. Analysiere Schadenmeldungen auf Inkonsistenzen. '
        'Gib strukturierten Fraud-Assessment-Report mit Plausibilitaetsbewertung 1-10 aus.'
    ),
}

def rag_retrieve(query, n=4):
    q_emb  = embedder.encode([query])
    scores = np.dot(chunk_embeddings, q_emb.T).squeeze()
    top    = np.argsort(scores)[::-1][:n]
    return [all_chunks[i] for i in top], [all_sources[i] for i in top]

def query_rag(incident, insurance_type, description='', role='expert', prompt_type='rag_expert'):
    docs, sources = rag_retrieve(f'{incident} {insurance_type} gedeckt')
    context       = '\n\n'.join(docs)

    if prompt_type == 'zero_shot':
        messages = [{'role': 'user', 'content':
            f"Ist '{incident}' bei '{insurance_type}' gedeckt? Antworte auf Deutsch."}]
    elif prompt_type == 'rag_basic':
        messages = [{'role': 'user', 'content':
            f'Versicherungsbedingungen:\n{context}\n\nIst "{incident}" bei "{insurance_type}" gedeckt?'}]
    else:
        if role == 'fraud_analyst':
            user_msg = (f'AVB:\n{context}\n\nFall: {insurance_type} | {incident}\n'
                        f'Beschreibung: {description or "Keine Angabe"}\n\n'
                        f'1. Plausibilitaetsbewertung (1-10)\n2. Auffaelligkeiten\n3. Empfehlung')
        elif role == 'customer_service':
            user_msg = f'AVB:\n{context}\n\nSchaden: {incident} | {insurance_type}\nErklaere freundlich.'
        else:
            user_msg = (f'AVB:\n{context}\n\nFall: {insurance_type} | {incident}\n'
                        f'1. Gedeckt? (Ja/Nein/Teilweise)\n2. AVB-Klausel\n3. Ausschluesse\n4. Naechste Schritte')
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPTS[role]},
            {'role': 'user',   'content': user_msg},
        ]

    resp = client_oai.chat.completions.create(
        model='gpt-4o-mini', messages=messages, temperature=0.1, max_tokens=500)
    return resp.choices[0].message.content, sources

print('RAG Pipeline konfiguriert.')

### C.3 Prompt-Vergleich und Rollen-Evaluation

In [ ]:
# Prompt-Varianten vergleichen
print('Prompt-Varianten Vergleich:')
print('=' * 60)
for pt in ['zero_shot', 'rag_basic', 'rag_expert']:
    answer, sources = query_rag('Kollisionsschaden Auffahrunfall', 'Vollkasko',
                                 prompt_type=pt)
    print(f'\nPrompt: {pt}')
    print(f'Quellen: {set(sources)}')
    print(f'Antwort: {answer[:200]}...')

# Rollen-Vergleich (Fraud Case)
print('\nRollen-Vergleich (Fraud Case):')
print('=' * 60)
for role in ['expert', 'customer_service', 'fraud_analyst']:
    answer, _ = query_rag(
        'Glasbruch Windschutzscheibe', 'Vollkasko',
        description='Kleiner Kratzer laut Kunde, kaum sichtbar',
        role=role, prompt_type='rag_expert')
    print(f'\nRolle: {role}')
    print(f'Antwort: {answer[:300]}...')

### C.4 End-to-End Evaluation (LM-as-Judge)

In [ ]:
TEST_CASES = [
    ('Parkschaden unbekannter Dritter',  'Vollkasko'),
    ('Glasbruch Windschutzscheibe',      'Teilkasko'),
    ('Hagelschaden Motorhaube',          'Teilkasko'),
    ('Vandalismusschaden Lack',          'Vollkasko'),
    ('Auffahrunfall Frontschaden',       'Vollkasko'),
    ('Diebstahl Fahrzeug',               'Teilkasko'),
    ('Kollisionsschaden Parkhaus',       'Haftpflicht'),
    ('Marderschaden Kabel',              'Teilkasko'),
    ('Brandschaden Motorraum',           'Vollkasko'),
    ('Kratzer Parkrempler',              'Haftpflicht'),
]

scores = {'zero_shot': 0, 'rag_basic': 0, 'rag_expert': 0}
print(f'LM-as-Judge Evaluation ({len(TEST_CASES)} Testfaelle x 3 Prompt-Varianten):')

for incident, ins_type in TEST_CASES:
    for pt in scores:
        answer, _ = query_rag(incident, ins_type, prompt_type=pt)
        judge = client_oai.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content':
                f'Bewerte (1-5): Ist "{incident}" bei "{ins_type}" gedeckt?\n'
                f'Antwort: {answer[:200]}\nNur Zahl 1-5.'}],
            temperature=0, max_tokens=3,
        )
        try:
            scores[pt] += int(judge.choices[0].message.content.strip())
        except:
            scores[pt] += 3

print('\nErgebnisse (Durchschnitt 1-5):')
for pt, total in scores.items():
    avg = total / len(TEST_CASES)
    print(f'  {pt:15}: {avg:.2f}/5.0')
    scores[pt] = avg

with open('models/rag_results.json', 'w') as f:
    json.dump({'prompt_scores': scores, 'n_test_cases': len(TEST_CASES)}, f, indent=2)
print('NLP Block abgeschlossen.')

---
## Ergebnisse sichern


In [ ]:
from google.colab import drive, files
import shutil

# Zusammenfassung
summary = {
    'ml_block': {
        'linear_regression': {'rmse': float(rmse_lr), 'r2': float(r2_lr)},
        'random_forest':     {'rmse': float(rmse_rf), 'r2': float(r2_rf)},
        'xgboost':           {'rmse': float(rmse_xgb), 'r2': float(r2_xgb)},
        'fraud_auc':         float(roc_auc_score(yf_test_labels, yf_prob)),
    },
    'cv_block': {
        'clip_zero_shot':    float(acc_clip),
        'transfer_learning': float(acc_tl),
        'fine_tuning':       float(acc_ft),
        'gpt4o_vision':      float(acc_gpt),
    },
    'nlp_block': dict(scores),
}
with open('models/final_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Alle Ergebnisse:')
print(f'  ML  — XGBoost: RMSE=CHF {rmse_xgb:.0f}, R2={r2_xgb:.3f}, Fraud AUC={summary["ml_block"]["fraud_auc"]:.3f}')
print(f'  CV  — Fine-Tuning: Accuracy={acc_ft:.2%}')
print(f'  NLP — RAG Expert: {scores.get("rag_expert", 0):.2f}/5.0')

# Google Drive Backup
drive.mount('/content/drive')
shutil.copytree('models', '/content/drive/MyDrive/InsuranceAI/models', dirs_exist_ok=True)
print('Modelle auf Google Drive gespeichert.')

# Lokaler Download
shutil.make_archive('insurance_models', 'zip', 'models')
files.download('insurance_models.zip')